# Splicing matrix: alternative splicing variants by sample

One objective: see whether samples fall into groups (endotypes) by the way they splice, using the long-read leafcutter results from S4. Each matrix is drawn with all samples and again without the samples named in `exclude`; a last view shows every splicing variant of the genes you name.

In [ ]:
source("src/heatmap.R")   # read_groups, show_heatmap
source("src/splicing.R")
source("src/paths.R")
loc <- run_locations()        # every folder of the run, from paths.json
comparison <- loc$comparison
exclude <- c("420_20190402")        # samples to leave out in the second drawing; character(0) skips it
genes <- c("HLA-B", "ERAP1", "ERAP2")   # genes to look at whether or not their clusters pass the test
groups <- read_groups(file.path(loc$leafcutter, paste0(comparison, ".lr_leafcutter.groups_file.txt")))

**Explanation.** The line that starts with `loc` reads `paths.json`, the one place where the run's results folder, the comparison name and the dataset name are set. The other cells use short names such as `loc$leafcutter`, so to look at another run you change `paths.json` and nothing in the notebook. leafcutter gathers introns that share splice sites into clusters; every intron in a cluster is one splicing variant of that spot in the gene. `exclude` names the samples to leave out in the second drawings (`character(0)` skips them), and `genes` the genes for the last view.

In [ ]:
subs <- read_subisoforms(file.path(loc$leafcutter, paste0(comparison, ".lr_leafcutter.subisoform_clusters.txt")))
sig  <- read_cluster_significance(file.path(loc$leafcutter, paste0(comparison, ".lr_leafcutter.ds_cluster_significance.txt")))
eff  <- read_effect_sizes(file.path(loc$leafcutter, paste0(comparison, ".lr_leafcutter.ds_effect_sizes.txt")))
table(sig$status)

**Explanation.** The table counts the splicing clusters by what leafcutter could do with them. Only the ones marked `Success` were tested between the groups and have a p value; the others had too little coverage or too few samples with reads. This is why the matrices below show fewer variants than there are in the data.

In [ ]:
all_set <- psi_set(subs, groups, sig, eff)
without  <- psi_set(subs, groups, sig, eff, exclude = exclude)
label <- paste(exclude, collapse = ", ")
c(variants = nrow(all_set$x$psi), samples = ncol(all_set$x$psi), samples_without = ncol(without$x$psi))

**Explanation.** For every sample and splicing variant we work out PSI (percent spliced in): the share of that cluster's reads that use the variant, from 0 (never) to 1 (always). The numbers above are the variants and samples in the matrix.

## Volcano of splicing variants

In [ ]:
suppressPackageStartupMessages({ library(ggplot2); library(ggrepel) })
options(repr.plot.width = 7, repr.plot.height = 5)
delta_name <- grep("^deltapsi_", names(eff), value = TRUE)[1]          # leafcutter names the column for the second group
vol <- all_set$x$info
vol <- vol[!is.na(vol$delta_psi) & !is.na(vol$adj_p), ]                # only variants in clusters leafcutter tested
vol$neglog10 <- -log10(pmax(vol$adj_p, 1e-300))
vol$significant <- vol$adj_p < 0.05
vol$label <- ifelse(vol$gene %in% genes | rank(vol$adj_p, ties.method = "first") <= 10, vol$gene, "")
c(tested_variants = nrow(vol), significant = sum(vol$significant))

**Explanation.** One point per splicing variant (intron) in a cluster that leafcutter tested. The change in PSI (`delta_psi`) is the variant's share of its cluster's reads in one group minus the other; the column name above says which group is first. The test p value belongs to the whole cluster, so every variant of a cluster has the same height. `significant` marks clusters with an adjusted p below 0.05. The labels are the ten smallest adjusted p values and the genes in `genes`. The last line counts the variants tested and those that pass.

In [ ]:
p <- ggplot(vol, aes(delta_psi, neglog10, colour = significant)) +
  geom_hline(yintercept = -log10(0.05), linetype = "dashed", colour = "grey50") +
  geom_vline(xintercept = 0, colour = "grey80") +
  geom_point(alpha = 0.7, size = 1.8) +
  geom_text_repel(aes(label = label), size = 3, max.overlaps = Inf, show.legend = FALSE) +
  scale_colour_manual(values = c("FALSE" = "grey60", "TRUE" = "#D55E00"), name = "adjusted p < 0.05") +
  labs(x = paste0("change in PSI (", sub("^deltapsi_", "", delta_name), " minus the other group)"),
       y = "-log10 adjusted p of the cluster", title = paste0("Splicing variants, ", comparison)) +
  theme_bw(base_size = 12) + theme(panel.grid = element_blank())
save_plot(p, "splicing_volcano")

**Explanation.** The volcano plot of splicing variants. Right and left show which group uses a variant more; higher means a smaller adjusted p for its cluster. The dashed line is adjusted p = 0.05. With few samples in one group the leafcutter test has little power, so a flat plot near the bottom is an honest result, and points above the line are leads to check in the PSI heatmap and in IGV. The same variant can appear in several genes' clusters, so look at the gene label together with the cluster.

In [ ]:
show_heatmap(splicing_heatmap(select_variants(all_set$x, alpha = 0.05), all_set$groups, title = "Clusters that differ between groups, all samples"), "splicing_heatmap")
if (length(exclude) > 0) show_heatmap(splicing_heatmap(select_variants(without$x, alpha = 0.05), without$groups, title = paste0("Clusters that differ between groups, without ", label)), "splicing_heatmap_without")

**Explanation.** Rows are the splicing variants (named by gene, cluster and the first and last splice position; a variant with several introns says how many) of the clusters leafcutter found different between the groups (adjusted p below 0.05); columns are samples split by group. Dark means the sample uses that variant often. The bar at the right is the change in PSI between the groups (second group minus first). Samples that clutter together by pattern, rather than by group, are candidate endotypes. Grey means the cluster had no reads in that sample. The second drawing is the same variants without the samples in `exclude`. The p values and deltaPSI bar are the pipeline's, computed with all samples; only the PSI shown changes, so to test the groups without a sample, rerun the multisample step without it.

In [ ]:
show_heatmap(splicing_heatmap(select_variants(all_set$x, alpha = 1), all_set$groups, title = "All tested splicing variants, all samples"), "splicing_heatmap_all")
if (length(exclude) > 0) show_heatmap(splicing_heatmap(select_variants(without$x, alpha = 1), without$groups, title = paste0("All tested splicing variants, without ", label)), "splicing_heatmap_all_without")

**Explanation.** The same matrix for every cluster that was tested, not only the ones that differ between the groups. This is where an endotype would show up as a split inside one group. Rows are named and the figure is as tall as the rows need; for this many rows the PDF in `figures/` is the one to zoom into.

In [ ]:
g <- select_genes(all_set$x, genes)
g$info[, c("gene", "cluster", "delta_psi", "adj_p")]

**Explanation.** Every splicing variant of the genes in `genes`, from clusters with at least two variants, whether or not leafcutter found the cluster different between the groups. A message above the table names any gene that is not in this run at all, or only in clusters with a single variant, where there is nothing to compare. The `adj_p` is empty for clusters leafcutter did not test.

In [ ]:
if (nrow(g$psi) > 0) {
  show_heatmap(splicing_heatmap(g, all_set$groups, title = paste0(paste(genes, collapse = ", "), ", all samples"), split_by_gene = TRUE), "splicing_heatmap_genes")
  if (length(exclude) > 0) {
    gw <- select_genes(without$x, genes)
    show_heatmap(splicing_heatmap(gw, without$groups, title = paste0(paste(genes, collapse = ", "), ", without ", label), split_by_gene = TRUE), "splicing_heatmap_genes_without")
  }
}

**Explanation.** The same variants as a PSI matrix, one block per gene, with the samples in columns split by group. Dark means the sample uses that variant often. The second drawing leaves out the samples in `exclude`.